## Loading requirements and dependencies

In [ ]:
!git clone https://github.com/mouvzee/MaskArchitectureAnomaly_Project2026.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if f.endswith(".bin") or f.endswith(".pth"):
            print(os.path.join(root, f))

In [ ]:
%cd /content/MaskArchitectureAnomaly_Project2026/eomt/

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd /content/MaskArchitectureAnomaly_Project2026/eval/

In [ ]:
!pip install ood-metrics

## Setup & Imports

In [ ]:
CITY_CKPT  = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"
COCO_CKPT  = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
DATA_PATH_CITY = "/content/drive/MyDrive/project_data_city"
DATA_PATH_COCO = "/content/drive/MyDrive/project_data_coco"
CITY_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
COCO_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
# ↑ point to the folder containing the zip, not the zip itself

In [ ]:
import yaml
import importlib
import warnings

import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
from torchmetrics import JaccardIndex

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

seed_everything(0, verbose=False)
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module`.*",
)

DEVICE     = 0          # indice GPU
IMG_IDX    = 0          # immagine da visualizzare

CITY_DATA_PATH = DATA_PATH_CITY
COCO_DATA_PATH = DATA_PATH_COCO
CITY_CKPT      = CITY_CKPT
COCO_CKPT      = COCO_CKPT

# Config YAML forniti dal laboratorio
CITY_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
COCO_CONFIG_PATH = "/content/MaskArchitectureAnomaly_Project2026/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
# ──────────────────────────────────────────────────────────────────────────────

IGNORE_INDEX           = 255
NUM_CITYSCAPES_CLASSES = 19

## Load Model and Data

In [ ]:
def load_model_and_data(config_path, ckpt_path, data_path, device):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    # Dataset
    data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
    data_module = getattr(importlib.import_module(data_module_name), class_name)
    data_kwargs = config["data"].get("init_args", {})
    data = data_module(
        path=data_path,
        batch_size=1,
        num_workers=0,
        check_empty_targets=False,
        **data_kwargs,
    ).setup()


    # Encoder
    enc_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_mod, enc_cls = enc_cfg["class_path"].rsplit(".", 1)
    encoder = getattr(importlib.import_module(enc_mod), enc_cls)(
        img_size=data.img_size, **enc_cfg.get("init_args", {})
    )

    # Network
    net_cfg = config["model"]["init_args"]["network"]
    net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = getattr(importlib.import_module(net_mod), net_cls)(
        masked_attn_enabled=False,
        num_classes=data.num_classes,
        encoder=encoder,
        **net_kwargs,
    )

    # Lightning module
    lit_mod, lit_cls = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_mod), lit_cls)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    model = lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    ).eval().to(device)

    # Weights
    ckpt = torch.load(ckpt_path, map_location=f"cuda:{device}", weights_only=False)
    model.load_state_dict(ckpt, strict=False)
    print(f"Loaded: {ckpt_path}")

    return model, data

In [ ]:
# ── Cityscapes ────────────────────────────────────────────────────────────────
city_model, city_data = load_model_and_data(
    config_path = CITY_CONFIG_PATH,
    ckpt_path   = CITY_CKPT,
    data_path   = DATA_PATH_CITY,
    device      = DEVICE,
)

# ── COCO ──────────────────────────────────────────────────────────────────────

coco_model, coco_data = load_model_and_data(
    config_path = COCO_CONFIG_PATH,
    ckpt_path   = COCO_CKPT,
    data_path   = DATA_PATH_COCO,
    device      = DEVICE,
)

# ── Fine-Tuned COCO model (city_data) ────────────────────────────────────────────────────────
finetuned_model, finetuned_data = load_model_and_data(
    config_path = CITY_CONFIG_PATH,  # Usa la stessa config di city_data
    ckpt_path   = COCO_CKPT,        # Ma i pesi del modello COCO fine-tuned su city_data
    data_path   = DATA_PATH_CITY,   # Usa i dati di city_data per la evaluation
    device      = DEVICE,
)


print(f"\ncity  →  img_size={city_data.img_size}, num_classes={city_data.num_classes}")
print(f"coco  →  img_size={coco_data.img_size}, num_classes={coco_data.num_classes}")
print(f"finetuned  →  img_size={finetuned_data.img_size}, num_classes={finetuned_data.num_classes}")

## Unzip Datasets

In [ ]:
!unzip "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets.zip" -d /content/datasets/

## EvalAnomalyEOMT.py

In [ ]:
# Cella 2: Funzione di Valutazione (V9 - Ground Truth Corretta)
import os
import glob
import torch
import numpy as np
from PIL import Image
import torch.nn.functional as F
from torchvision.transforms import Compose, ToTensor
from tqdm import tqdm

from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr

def evaluate_anomaly_eomt(model, data, images_dir, gt_dir, device='cuda:0'):
    # Nessun Resize! Manteniamo le proporzioni originali (es. 1280x720)
    input_transform = Compose([ToTensor()])

    model.eval()

    ood_gts_list = []
    anomaly_scores = {
        "MSP": [],
        "Max_Entropy": [],
        "MaxLogit": [],
        "RbA": []
    }

    image_paths = glob.glob(os.path.join(images_dir, '*.png'))
    image_paths.extend(glob.glob(os.path.join(images_dir, '*.jpg')))

    print(f"Inizio inferenza su {len(image_paths)} immagini (Risoluzione Naturale)...")

    for img_path in tqdm(image_paths):
        # =============================================================
        # 1. GROUND TRUTH (Logica Originale Funzionante)
        # =============================================================
        img_basename = os.path.basename(img_path)
        gt_path = os.path.join(gt_dir, img_basename.replace('.jpg', '.png'))

        if not os.path.exists(gt_path):
            continue

        gt_img = Image.open(gt_path).convert('L')
        gt_array = np.array(gt_img) # Nessun Resize anche qui!

        # Ripristinato il blocco di mappatura originale (0 = Strada, 1 = OOD)
        if "RoadAnomaly" in gt_path:
            gt_array = np.where((gt_array==2), 1, gt_array)
        elif "LostAndFound" in gt_path:
            gt_array = np.where((gt_array==0), 255, gt_array)
            gt_array = np.where((gt_array==1), 0, gt_array)
            gt_array = np.where((gt_array>1) & (gt_array<201), 1, gt_array)
        elif "Streethazard" in gt_path:
            gt_array = np.where((gt_array==14), 255, gt_array)
            gt_array = np.where((gt_array<20), 0, gt_array)
            gt_array = np.where((gt_array==255), 1, gt_array)

        if 1 not in np.unique(gt_array):
            continue

        ood_gts_list.append(gt_array.flatten())

        # =============================================================
        # 2. INFERENZA E WINDOWING
        # =============================================================
        image = Image.open(img_path).convert('RGB')
        img_tensor = input_transform(image)
        img_tensor = (img_tensor * 255).to(torch.uint8)

        with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            imgs = [img_tensor.to(device)]
            img_sizes = [imgs[0].shape[-2:]]

            crops, origins = model.window_imgs_semantic(imgs)
            mask_logits_per_layer, class_logits_per_layer = model(crops)

            # ---> FIX SPAZIALE: Interpoliamo in base al CROP (es. 512), NON all'immagine globale!
            mask_logits = F.interpolate(mask_logits_per_layer[-1], size=crops.shape[-2:], mode="bilinear", align_corners=False)
            class_logits = class_logits_per_layer[-1]

            # =============================================================
            # 3. METRICHE STANDARD (FINAL FIX)
            # =============================================================
            crop_probs = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
            probs_list = model.revert_window_logits_semantic(crop_probs, origins, img_sizes)

            # Converti in float32 per precisione numerica (Logit grezzi)
            probs_f32 = probs_list[0].float().cpu().numpy()   # (C, H, W)

            # --- MaxLogit ---
            # Usa i punteggi grezzi
            maxlogit_score = 1.0 - np.max(probs_f32, axis=0)
            anomaly_scores["MaxLogit"].append(maxlogit_score.flatten())

            # --- Softmax Scaling (Base per MSP e MaxEntropy) ---
            e_x = np.exp(probs_f32 - np.max(probs_f32, axis=0, keepdims=True))
            softmax_probs = e_x / np.sum(e_x, axis=0, keepdims=True)

            # --- MSP (Maximum Softmax Probability, t=1.0) ---
            msp_score = 1.0 - np.max(softmax_probs, axis=0)
            anomaly_scores["MSP"].append(msp_score.flatten())

            # --- MaxEntropy ---
            # CALCOLATA SULLE SOFTMAX_PROBS! 
            # Ora i pixel OOD avranno una distribuzione piatta e quindi un'entropia altissima.
            entropy_score = -np.sum(softmax_probs * np.log(softmax_probs + 1e-12), axis=0)
            anomaly_scores["Max_Entropy"].append(entropy_score.flatten())

            # =============================================================
            # 4. VERO RbA (Log-Sum Anti-Underflow)
            # =============================================================
            B, Q = mask_logits.shape[0], mask_logits.shape[1] 
            
            mask_probs_f32 = mask_logits.sigmoid().float()
            class_probs_f32 = class_logits.softmax(dim=-1).float()

            prob_known_q = 1.0 - class_probs_f32[..., -1]
            joint_probs = mask_probs_f32 * prob_known_q.view(B, Q, 1, 1)

            rejection_probs = torch.clamp(1.0 - joint_probs, min=1e-7, max=1.0)
            log_rba_crop = torch.sum(torch.log(rejection_probs), dim=1, keepdim=True).half()

            rba_list = model.revert_window_logits_semantic(log_rba_crop, origins, img_sizes)
            rba_score = rba_list[0].squeeze(0).cpu().numpy()

            anomaly_scores["RbA"].append(rba_score.flatten())

            del image, img_tensor, imgs, crops, mask_logits, class_logits, crop_probs, log_rba_crop
            torch.cuda.empty_cache()

    # =============================================================
    # VALUTAZIONE FINALE
    # =============================================================
    print("\nElaborazione dati e calcolo AUC...")
    ood_gts = np.concatenate(ood_gts_list)
    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    for k in anomaly_scores:
        if len(anomaly_scores[k]) > 0:
            anomaly_scores[k] = np.concatenate(anomaly_scores[k])

    def eval_metric(scores_array, name):
        val_out = np.concatenate((scores_array[ind_mask], scores_array[ood_mask]))
        val_label = np.concatenate((np.zeros(ind_mask.sum()), np.ones(ood_mask.sum())))

        prc_auc = average_precision_score(val_label, val_out)
        fpr = fpr_at_95_tpr(val_out, val_label)

        print(f"[{name}] AUPRC: {prc_auc*100.0:.2f} | FPR@TPR95: {fpr*100.0:.2f}")

    print("\n" + "="*40)
    print("       RISULTATI ANOMALY DETECTION")
    print("="*40)

    for metric_name, scores_array in anomaly_scores.items():
        if len(scores_array) > 0:
            eval_metric(scores_array, metric_name)

## Cityscapes Model

### RoadAnomaly

In [ ]:
evaluate_anomaly_eomt(
    model=city_model,
    data=city_data,
    images_dir='/content/datasets/Validation_Dataset/RoadAnomaly/images',
    gt_dir='/content/datasets/Validation_Dataset/RoadAnomaly/labels_masks',
    device='cuda:0'
)

### SMIYC RoadAnomaly-21

### SMIYC RoadObstacle-21

### FS Lost&Found

### FS Static

## COCO Model

### RoadAnomaly

### SMIYC RoadAnomaly-21

### SMIYC RoadObstacle-21

### FS Lost&Found

### FS Static

## Finetuned Model

### RoadAnomaly

### SMIYC RoadAnomaly-21

### SMIYC RoadObstacle-21

### FS Lost&Found

### FS Static